# 🧠 PINKCC : Entraînement ResNet18 (Méthode B + Advanced Augmentation)

Ce notebook teste la robustesse du modèle en combinant la correction du déséquilibre des classes (**Méthode B** via Weighted Sampler) avec notre nouveau pipeline d'augmentation de données avancé (flou, contraste, rotation).

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import matplotlib.pyplot as plt
import sys
import copy
from torch.utils.data import Subset

# --- 1. CONFIGURATION ROBUSTE DU CHEMIN (Compatible Colab & Papermill) ---
cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent

if str(project_root / "src") not in sys.path:
    sys.path.append(str(project_root / "src"))

# --- 2. IMPORTS DES MODULES PINKCC ---
from pinkcc_ct_seg.data.dataset import BrainMRIDataset
from pinkcc_ct_seg.data.split import make_train_val_split
from pinkcc_ct_seg.data.loaders import make_loaders
from pinkcc_ct_seg.models.resnet18 import build_resnet18
from pinkcc_ct_seg.training.engine import train_one_epoch, validate_one_epoch
from pinkcc_ct_seg.evaluation.metrics import compute_metrics

# --- 3. CONFIGURATION MATÉRIELLE ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"💻 Appareil utilisé : {device}")

💻 Appareil utilisé : cpu


### 1. Préparation des données et Split Stratifié

In [2]:
from pinkcc_ct_seg.data.transforms import get_train_transforms, get_eval_transforms
from torch.utils.data import Subset

data_dir = project_root / "data" / "raw" / "brain_mri" / "Training"

# Étape A : Instanciation du Dataset global pour l'exploration du dossier
full_dataset = BrainMRIDataset(data_dir)

# Étape B : Extraction des labels pour le split stratifié
all_labels = [label for _, label in full_dataset.samples]

# Étape C : Calcul des indices via votre module de split
train_idx, val_idx = make_train_val_split(all_labels, val_size=0.2, random_state=42)

# Étape D : CORRECTION ! On applique les transformations ici, directement sur la base
train_dataset_base = BrainMRIDataset(
    data_dir, 
    transform=get_train_transforms(augmentation_level='advanced')
)
val_dataset_base = BrainMRIDataset(
    data_dir, 
    transform=get_eval_transforms()
)

# Étape E : Création des Subsets PyTorch
train_subset = Subset(train_dataset_base, train_idx)
val_subset = Subset(val_dataset_base, val_idx)

# Étape F : Création des DataLoaders 
train_loader, val_loader = make_loaders(
    train_subset, 
    val_subset, 
    batch_size=32, 
    weighted=True, 
    augmentation_level='advanced'
)

print(f"✅ DataLoaders prêts. Entraînement: {len(train_subset)} images | Validation: {len(val_subset)} images.")

✅ DataLoaders prêts. Entraînement: 2296 images | Validation: 574 images.


### 2. Initialisation du Modèle

In [3]:
# INITIALISATION
model = build_resnet18(num_classes=2, pretrained=True).to(device)

# On utilise une perte standard car le déséquilibre est géré par le Sampler (Méthode B)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

✅ ResNet18 chargé avec poids préentraînés.


### 3. Boucle d'entraînement

In [4]:
EPOCHS = 10
train_losses, val_losses = [], []
best_val_loss = float('inf')

# S'assurer que le dossier de sauvegarde existe
models_dir = project_root / "models"
models_dir.mkdir(exist_ok=True)
save_path = models_dir / "best_resnet18_methodB_advanced.pt"

print(" Début de l'entraînement...")

for epoch in range(EPOCHS):
    print(f"\n--- Époque {epoch+1}/{EPOCHS} ---")
    
    # On récupère le dictionnaire de résultats pour l'entraînement
    train_results = train_one_epoch(model, train_loader, criterion, optimizer, device)
    
    # On récupère le dictionnaire de résultats pour la validation
    val_results = validate_one_epoch(model, val_loader, criterion, device)
    
    # Extraction des valeurs pour le suivi
    train_losses.append(train_results["loss"])
    val_losses.append(val_results["loss"])
    
    print(f" Train Loss: {train_results['loss']:.4f} | Val Loss: {val_results['loss']:.4f}")
    print(f" Train Acc: {train_results['acc']:.4f} | Val Acc: {val_results['acc']:.4f}")
    
    # Sauvegarde basée sur la perte de validation
    if val_results["loss"] < best_val_loss:
        best_val_loss = val_results["loss"]
        torch.save(model.state_dict(), save_path)
        print(" Nouveau meilleur modèle sauvegardé !")

 Début de l'entraînement...

--- Époque 1/10 ---


 Train Loss: 0.2679 | Val Loss: 0.1984
 Train Acc: 0.9033 | Val Acc: 0.9024
 Nouveau meilleur modèle sauvegardé !

--- Époque 2/10 ---


 Train Loss: 0.1660 | Val Loss: 0.1214
 Train Acc: 0.9421 | Val Acc: 0.9547
 Nouveau meilleur modèle sauvegardé !

--- Époque 3/10 ---


 Train Loss: 0.1037 | Val Loss: 0.2031
 Train Acc: 0.9678 | Val Acc: 0.9321

--- Époque 4/10 ---


 Train Loss: 0.0996 | Val Loss: 0.5074
 Train Acc: 0.9665 | Val Acc: 0.8206

--- Époque 5/10 ---


...

 Train Loss: 0.0847 | Val Loss: 0.4833
 Train Acc: 0.9734 | Val Acc: 0.8484

--- Époque 7/10 ---


 Train Loss: 0.0707 | Val Loss: 0.2695
 Train Acc: 0.9774 | Val Acc: 0.9321

--- Époque 8/10 ---


 Train Loss: 0.0560 | Val Loss: 0.1249
 Train Acc: 0.9839 | Val Acc: 0.9477

--- Époque 9/10 ---


 Train Loss: 0.0633 | Val Loss: 0.0535
 Train Acc: 0.9795 | Val Acc: 0.9721
 Nouveau meilleur modèle sauvegardé !

--- Époque 10/10 ---


 Train Loss: 0.0506 | Val Loss: 0.0801
 Train Acc: 0.9852 | Val Acc: 0.9808


### 4. Évaluation Finale des Performances

In [5]:
# Recharger le meilleur modèle
model.load_state_dict(torch.load(save_path))

# Exécuter une dernière validation pour obtenir les prédictions finales
final_results = validate_one_epoch(model, val_loader, criterion, device)

# Appel de votre fonction compute_metrics avec les bonnes clés du dictionnaire
# y_true = targets, y_pred = preds
metrics_results = compute_metrics(
    y_true=final_results["targets"], 
    y_pred=final_results["preds"]
)

print("\n --- RÉSULTATS GLOBAUX --- ")
print(f"Précision (Accuracy) : {metrics_results['accuracy']:.4f}")
print(f"F1-Score             : {metrics_results['f1']:.4f}")
print(f"Rappel (Recall)      : {metrics_results['recall']:.4f}")


 --- RÉSULTATS GLOBAUX --- 
Précision (Accuracy) : 0.9721
F1-Score             : 0.9839
Rappel (Recall)      : 0.9859
